# Biofilter — Report: Variant Modeling

Map an input list of variants (rsID or chr:pos) to biologically connected **variant×variant pairs**,
where both variants in every pair come from the input.

```
Input variants (rsID or chr:pos)
    ↓  DB lookup + window_bp
Genes overlapping input variants
    ↓  group membership (Pathway, GO, Disease, …)
Groups
    ↓  co-membership → Gene×Gene pairs
Gene×Gene pairs  [weight = # shared groups]
    ↓  cartesian of input variants per gene
Variant×Variant pairs  ← output
```

`group_support_count` is the biological weight: how many distinct groups connect the two genes.

See the explain guide: `biofilter/modules/report/reports_explain/report_variant_modeling.md`

### 1. Start Biofilter

In [ ]:
from biofilter import Biofilter

bf = Biofilter(debug_mode=False)
bf

### 2. Inspect report metadata

In [ ]:
report_name = 'variant_modeling'

print('name:', report_name)
print('\navailable columns:')
print(bf.report.available_columns(report_name))

print('\nexample_input:')
print(bf.report.example_input(report_name))

print('\nexplain:')
print(bf.report.explain(report_name))

### 3. Basic run — rsID inputs, Pathway grouping

Input: four variants from the APOE region and PCSK9.  
Grouping: Pathway (Reactome).  
Both variants in every output pair are from the input list.

In [ ]:
df = bf.report.run(
    'variant_modeling',
    input_data=[
        'rs429358',       # APOE ε4
        'rs7412',         # APOE ε2
        'rs2479409',      # PCSK9 promoter
        'rs11591147',     # PCSK9 R46L (loss-of-function)
    ],
    build=38,
    window_bp=0,
    group_entity_groups=['Pathway'],
)

print(f'Pairs: {len(df):,}')
df.head(20)

#### 3b. Top pairs by biological weight

In [ ]:
cols = [
    'variant_1_rsid', 'gene_1_name',
    'variant_2_rsid', 'gene_2_name',
    'group_support_count', 'group_support_names',
    'data_source_support_names',
]

df[cols].sort_values('group_support_count', ascending=False).head(20)

### 4. Mixed input — rsID and chr:pos

The report accepts both formats in the same list.

In [ ]:
df_mixed = bf.report.run(
    'variant_modeling',
    input_data=[
        'rs429358',           # rsID
        'chr19:44908684',     # chr:pos (APOE region)
        '2:21044574',         # bare chr:pos (APOE2D region)
    ],
    build=38,
    group_entity_groups=['Pathway'],
)

print(f'Pairs: {len(df_mixed):,}')
df_mixed[cols].head(20)

### 5. Multiple group types — Pathway + GO + Disease

Using multiple group types increases `group_support_count` when genes share more than one biological context.

In [ ]:
df_multi = bf.report.run(
    'variant_modeling',
    input_data=[
        'rs429358',
        'rs7412',
        'rs2479409',
        'rs11591147',
    ],
    build=38,
    group_entity_groups=['Pathway', 'GO', 'Disease'],
)

print(f'Pairs: {len(df_multi):,}')
df_multi[cols].sort_values('group_support_count', ascending=False).head(20)

#### 5b. group_support_count distribution

In [ ]:
import matplotlib.pyplot as plt

if not df_multi.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    df_multi['group_support_count'].value_counts().sort_index().plot(
        kind='bar', ax=ax, color='steelblue', edgecolor='white'
    )
    ax.set_xlabel('group_support_count (weight)')
    ax.set_ylabel('# variant pairs')
    ax.set_title('Biological weight distribution across variant pairs')
    plt.tight_layout()
    plt.show()

#### 5c. Gene pair heatmap (group_support_count)

In [ ]:
import pandas as pd

if not df_multi.empty:
    gene_pair_weight = (
        df_multi.groupby(['gene_1_name', 'gene_2_name'])['group_support_count']
        .max()
        .reset_index()
    )

    pivot = gene_pair_weight.pivot(index='gene_1_name', columns='gene_2_name', values='group_support_count').fillna(0)

    fig, ax = plt.subplots(figsize=(max(6, len(pivot.columns)), max(4, len(pivot))))
    im = ax.imshow(pivot.values, aspect='auto', cmap='Blues')
    plt.colorbar(im, ax=ax, label='group_support_count')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_yticks(range(len(pivot)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
    ax.set_yticklabels(pivot.index)
    ax.set_title('Gene pair biological weight (max group_support_count)')
    plt.tight_layout()
    plt.show()

    print('Gene pair summary:')
    display(gene_pair_weight.sort_values('group_support_count', ascending=False))

### 6. Restrict to a specific data source

Use `group_data_sources` to filter group membership to a single source (e.g., Reactome only).

In [ ]:
df_reactome = bf.report.run(
    'variant_modeling',
    input_data=['rs429358', 'rs7412', 'rs2479409', 'rs11591147'],
    group_entity_groups=['Pathway'],
    group_data_sources=['Reactome'],
)

print(f'Reactome-only pairs: {len(df_reactome):,}')
df_reactome[cols].head(20)

### 7. Window extension

`window_bp` extends gene boundaries when assigning variants to genes.
Useful when variants fall in regulatory regions near gene loci.

In [ ]:
results = {}
for window in [0, 5_000, 25_000]:
    df_w = bf.report.run(
        'variant_modeling',
        input_data=['rs429358', 'rs7412', 'rs2479409', 'rs11591147'],
        group_entity_groups=['Pathway'],
        window_bp=window,
    )
    results[window] = len(df_w)
    print(f'window_bp={window:>6,}  →  {len(df_w):,} pairs')

### 8. Input from file

Pass a path to a plain-text file (one rsID or chr:pos per line).

In [ ]:
from pathlib import Path

# Create a temporary input file for the tutorial
tmp_dir = Path('tmp/variant_modeling_tutorial')
tmp_dir.mkdir(parents=True, exist_ok=True)

input_file = tmp_dir / 'variants.txt'
input_file.write_text('rs429358\nrs7412\nrs2479409\nrs11591147\n')

df_file = bf.report.run(
    'variant_modeling',
    input_data=str(input_file),
    group_entity_groups=['Pathway'],
)

print(f'Pairs from file: {len(df_file):,}')
df_file[cols].head(10)

### 9. Safety check — max_pairs

The report estimates pair count before materialising. If the estimate exceeds `max_pairs` it aborts safely.

In [ ]:
df_safe = bf.report.run(
    'variant_modeling',
    input_data=['rs429358', 'rs7412', 'rs2479409', 'rs11591147'],
    group_entity_groups=['Pathway', 'GO', 'Disease'],
    max_pairs=5,   # intentionally low to trigger the check
)

if 'resolution_status' in df_safe.columns:
    print('Safety abort triggered:')
    print(df_safe[['resolution_status', 'estimated_pairs', 'max_pairs', 'suggestion']].to_string())
else:
    print(f'{len(df_safe):,} pairs — no abort')

### 10. Export results

In [ ]:
output_path = tmp_dir / 'variant_modeling_pairs.csv'
df_multi.to_csv(output_path, index=False)
print(f'Saved {len(df_multi):,} pairs → {output_path}')

### 11. Real cohort template

Replace the input list with your study variants and adjust group filters.

In [ ]:
# Option A: explicit list
my_variants = [
    'rs429358',
    'rs7412',
    # ... add your variants
]

# Option B: load from file
# my_variants = '/path/to/variants.txt'

# df_cohort = bf.report.run(
#     'variant_modeling',
#     input_data=my_variants,
#     build=38,
#     window_bp=0,
#     group_entity_groups=['Pathway', 'GO'],
#     group_data_sources=['Reactome'],
#     max_pairs=1_000_000,
# )

# print(f'Cohort pairs: {len(df_cohort):,}')
# df_cohort.to_csv('outputs/variant_modeling_cohort.csv', index=False)
# df_cohort.head(20)